In [ ]:
!mkdir /kaggle/temp

In [ ]:
!wget -O /kaggle/temp/Miniforge3.sh "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-$(uname)-$(uname -m).sh"

In [ ]:
!rm -rf /kaggle/temp/conda
!bash /kaggle/temp/Miniforge3.sh -b -p /kaggle/temp/conda

In [ ]:
import subprocess
subprocess.run(['/kaggle/temp/conda/bin/mamba', 'install', '-y', 'python=3.13', 'cuda-version=13.0', 'pytorch=2.11',
                'pillow', 'matplotlib', 'scipy', 'nvimgcodec', 'nccl', 'cuda-cudart-dev'], check=True)
subprocess.run(['/kaggle/temp/conda/bin/python', '-c',
                'import sys, torch, triton; print(sys.version); print(torch.__version__, torch.version.cuda, torch.cuda.nccl.version(), triton.__version__)'],
               check=True)

In [ ]:
!rm -rf /kaggle/temp/music_deep
!git clone -b exp-043-patch-set https://github.com/kwon-young/music_deep.git /kaggle/temp/music_deep
!git -C /kaggle/temp/music_deep log --oneline -1

In [ ]:
import os

# Exp 043 (patch-set assignment, PATCH_SET_ASSIGNMENT.md), same recipe and init as Exp 038.
# TEST = True: a run capped below 1 h (3 epochs under `timeout 35m`, inference on 30 pages under
# `timeout 8m` each) to check DDP + compile on T4, measure s/epoch and get a checkpoint.
# TEST = False: the full run. The test run measured 426 s/epoch (2xT4), 4.7 min of setup and
# compile, and ~7 s/page (association) / ~4 s/page (control) of inference, so 83 epochs (~589 min)
# under `timeout 10h` plus 351-page inference under 50 + 35 min stay below 11.5 h in the worst case.
# inference_coco.py writes its JSON at the end, so its timeouts keep a margin.
TEST = False
EPOCHS = 3 if TEST else 83
EXP_DIR = 'experiments/043_patch_set_test_1h' if TEST else 'experiments/043_patch_set'
TIMEOUT = '35m' if TEST else '10h'
INFER_TIMEOUT_ASSOC = '8m' if TEST else '50m'
INFER_TIMEOUT_CONTROL = '8m' if TEST else '35m'
INFER_PAGES = '--num_samples 30' if TEST else ''

# The COCO JSON is parsed fresh (~1 min) with the current code.
os.makedirs('/kaggle/temp/cache', exist_ok=True)

# Exp 037 detector + its Phase A refiner, from the private dataset kwonyoungchoi/music-deep-038-init
# (a dataset rather than a model: `kaggle kernels push` keeps dataset sources but drops model sources).
print(sorted(os.listdir('/kaggle/input')))
wanted = {'init_037.pt': '/kaggle/temp/init_037.pt', 'refiner_037_phase_a.pt': '/kaggle/temp/refiner_037_phase_a.pt'}
found = {name: [] for name in wanted}
for root, dirs, files in os.walk('/kaggle/input'):
    dirs[:] = [d for d in dirs if d not in ('ssl-imslp', 'trompa-coco', 'trompa-coco-cache')]
    for name in files:
        if name in wanted:
            found[name].append(os.path.join(root, name))
print(found)
for name, paths in found.items():
    assert len(paths) == 1, (name, paths)
    if os.path.lexists(wanted[name]):
        os.remove(wanted[name])
    os.symlink(paths[0], wanted[name])

In [ ]:
# Exp 043: Exp 038's recipe (Exp 037 + Phase A refiner in the step) with the patch-set assignment:
# K = 8 slots with a presence logit (the K = 5 checkpoint is remapped), voters in every patch within
# 0.5 patch of a GT, head output layers at 10x LR. Checkpoints every epoch. expandable_segments:
# the test run's OOM showed 4 GB reserved but unallocated (fragmentation).
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True PYTHONPATH=/kaggle/temp/music_deep /kaggle/temp/conda/bin/mamba run timeout {TIMEOUT} torchrun --nproc_per_node=2 /kaggle/temp/music_deep/src/train_detection.py \
    --exp_dir {EXP_DIR} \
    --patch_size 64 \
    --epochs {EPOCHS} \
    --anno_path ../input/datasets/kwonyoungchoi/trompa-coco/annotations/instances_trainval2017.json \
    --img_dir ../input/datasets/kwonyoungchoi/trompa-coco/trainval2017 \
    --headless \
    --cache_dir /kaggle/temp/cache/ \
    --use_sdpa \
    --log_epoch_interval 1 \
    --warmup_epochs 1 \
    --lr 1e-4 \
    --compile \
    --extract_patch_min 54 \
    --extract_patch_max 74 \
    --class_weight_variance_lambda 0.5 \
    --class_weight_size_difficulty_max 8.0 \
    --detector_checkpoint /kaggle/temp/init_037.pt \
    --line_refiner \
    --refiner_init /kaggle/temp/refiner_037_phase_a.pt \
    --refiner_lr_scale 1.0 \
    --assignment patch_set \
    --num_shapes 8 \
    --head_lr_scale 10

In [ ]:
# All 351 pages with the patch-set pipeline: presence floor, refine, association (symbols with
# cannot-link), support x quality. Evaluate locally (pycocotools is not installed here).
!PYTHONPATH=/kaggle/temp/music_deep/src /kaggle/temp/conda/bin/mamba run timeout {INFER_TIMEOUT_ASSOC} python /kaggle/temp/music_deep/src/inference_coco.py \
    --checkpoint {EXP_DIR}/train_detection/checkpoints/latest_model.pt \
    --refiner_checkpoint {EXP_DIR}/train_detection/checkpoints/latest_model.pt \
    --output_dir {EXP_DIR}/inference_association \
    --association \
    --anno_path ../input/datasets/kwonyoungchoi/trompa-coco/annotations/instances_trainval2017.json \
    --img_dir ../input/datasets/kwonyoungchoi/trompa-coco/trainval2017 \
    --cache_dir /kaggle/temp/cache/ \
    --patch_size 64 \
    --use_sdpa {INFER_PAGES}

In [ ]:
# Control on the same checkpoint: the Exp 035 line pipeline (refine -> NMS 4 px -> noisy-OR) and
# top-k symbols, without association.
!PYTHONPATH=/kaggle/temp/music_deep/src /kaggle/temp/conda/bin/mamba run timeout {INFER_TIMEOUT_CONTROL} python /kaggle/temp/music_deep/src/inference_coco.py \
    --checkpoint {EXP_DIR}/train_detection/checkpoints/latest_model.pt \
    --refiner_checkpoint {EXP_DIR}/train_detection/checkpoints/latest_model.pt \
    --output_dir {EXP_DIR}/inference_full_pipeline \
    --anno_path ../input/datasets/kwonyoungchoi/trompa-coco/annotations/instances_trainval2017.json \
    --img_dir ../input/datasets/kwonyoungchoi/trompa-coco/trainval2017 \
    --cache_dir /kaggle/temp/cache/ \
    --patch_size 64 \
    --use_sdpa {INFER_PAGES}